# REPAIR初始预算估算

## tl;dr
候选：B_loss=1.50 USDC/轮，B_wait=0.15 USDC/次等待的标准差代理。仅供Shadow/回放；不是已验证盈利参数。


## Context & Methods

目标是在Buy Only、Maker Only、15 USDC Session额度下选择初始风险预算。数据来自本地真实journal和Shadow输出。

### Key Assumptions
固定5股、局部方差不变；仅用15≤T<240基础槽位，排除DECISION重复粒度及UNKNOWN。波动来自10月14–15日，开仓成本来自10月17日，不能合并解释为同一策略效果。


In [1]:
from pathlib import Path
import importlib.util, json
project = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'src/polymaker').is_dir())
source = project/'docs/development/reviews/20261018_repair_initial_budgets/analyze.py'
spec = importlib.util.spec_from_file_location('repair_initial_budget_analysis', source)
module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(module)
evidence = module.analyze()


## Data
47场历史Shadow；最新13轮开仓成本逐笔回到原journal核验。完整源文件哈希、筛选口径与局限在同目录evidence.json。


In [2]:
print(json.dumps(evidence['scope'], ensure_ascii=False, indent=2))
print(json.dumps(evidence['coverage'], ensure_ascii=False, indent=2))


{
  "start": "2026-10-14T12:55:00+08:00",
  "last_market_start": "2026-10-15T23:05:00+08:00",
  "filter": "15 <= elapsed_sec < 240; exclude DECISION; volatility and probability VALID",
  "grain": "market / session / base risk_snapshot_id",
  "fixed_inventory_shares": 5,
  "timezone": "Asia/Shanghai"
}
{
  "base_slots_pre240": 10650,
  "unknown_or_invalid_vol_probability": 2061,
  "excluded_decision_records": 299,
  "valid_vol_probability_slots": 8589,
  "valid_actual_nonzero_inventory_slots": 1777,
  "runs": 47,
  "unique_markets": 47
}


## Results
### 等待预算尺度
这些是基于真实V_hat的固定5股场景，不是实测亏损分布。数值适合用精确值表比较；会话图展示其分位数。


In [3]:
print('1秒波动标准差代理:', json.dumps(evidence['fixed5_sd_1s'], ensure_ascii=False))
for row in evidence['wait_candidates']:
    print(row)
print('按市场中位数的敏感性:', evidence['market_median_fixed5_sd_1s'])


1秒波动标准差代理: {"n": 8589, "minimum": 0.0, "maximum": 0.4020477280734414, "p10": 0.04548621941384347, "p25": 0.07599672648869472, "p50": 0.11216526805654674, "p75": 0.15242586625073506, "p90": 0.19729889323440206, "p95": 0.230808760083368}
{'budget': 0.1, 'median_h_seconds': 0.7948463840321415, 'fraction_allowing_1s': 0.41436721387821634, 'fraction_allowing_2s': 0.22016532774478983, 'p10_h_seconds': 0.2568920734952124}
{'budget': 0.15, 'median_h_seconds': 1.7884043640723182, 'fraction_allowing_1s': 0.7383863080684596, 'fraction_allowing_2s': 0.4539527302363488, 'p10_h_seconds': 0.5780071653642279}
{'budget': 0.2, 'median_h_seconds': 3.179385536128566, 'fraction_allowing_1s': 0.9058097566655023, 'fraction_allowing_2s': 0.6899522645243916, 'p10_h_seconds': 1.0275682939808497}
按市场中位数的敏感性: {'n': 47, 'minimum': 0.01888771173179791, 'maximum': 0.1924257688366053, 'p10': 0.05711712494304795, 'p25': 0.07879342486045594, 'p50': 0.10975214582276141, 'p75': 0.1326796192939582, 'p90': 0.15293753231826

### 持仓损失预算
下列人数是初始开仓成本通过单一预算的轮次；不是HOLD推荐数量或新策略胜率。


In [4]:
print('初始开仓成本:', evidence['latest_entries']['initial_costs'])
for row in evidence['latest_entries']['loss_candidates']:
    print(row)


初始开仓成本: {'n': 13, 'minimum': 1.2, 'maximum': 4.55, 'p10': 1.48, 'p25': 2.1, 'p50': 2.6, 'p75': 3.3, 'p90': 3.6500000000000004, 'p95': 4.039999999999999}
{'budget': 1.0, 'below_budget': 0, 'total': 13, 'above_budget': 13}
{'budget': 1.5, 'below_budget': 2, 'total': 13, 'above_budget': 11}
{'budget': 2.0, 'below_budget': 3, 'total': 13, 'above_budget': 10}


## Takeaways
1.50让13轮中2轮通过初始本金损失检查；1.00为0轮，2.00为3轮。0.15等待预算在5股中位波动下允许约1.79秒，在P90波动尺度下约0.58秒。实际敞口须重新计算。没有检验新策略最终结算收益，不能将预算候选解释为盈利保证。
